# GFP 결합체 설계와 형광 변화 평가 · v0.1.9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/GFP_Binder_T4.ipynb?v=0.1.9)

GFP 결합체의 뼈대와 서열을 설계하고 복합체 구조·형광 변화 가능성을 평가합니다. 대조군 실험으로 새 후보를 평가할 조건의 신뢰도를 확인합니다. T4 GPU 런타임에서 모드를 선택하고 전체 셀을 실행합니다.

| RUN_MODE | 실행 범위 |
| --- | --- |
| full | RFD3 → LigandMPNN → RF3 → 형광 평가 |
| benchmark | 알려진 두 대조군의 결합 자세 재현 |
| followup | 실행 05의 시드 확대·템플릿 제거 조건 재실행 |
| diagnostics | enhancer 단독 접힘·복합체 배치 분리, minimizer MEGADOCK GPU 도킹 |

기본값은 `diagnostics`입니다. Enhancer는 NanoBodyBuilder2로 단독 구조를 한 번 예측하고, RF3의 3개 시드에서 예측 구조와 결정 구조 조건을 비교합니다. Minimizer는 MEGADOCK GPU 강체 도킹 후 점수 상위 최대 100개 자세를 평가합니다. 최초 실행에서 CUDA 프로그램을 빌드합니다. v0.1.9의 전체 실험 소요시간은 아직 측정하지 않았습니다.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")
sys.path.insert(0, str(checkout / "gfp_pipeline"))
from notebook_workflow import build_jobs, run_jobs, display_results, package_results


## 실행 설정

- `diagnostics`: `DIAGNOSTIC_EXPERIMENT`에서 `all`, `enhancer_split`, `minimizer_docking`을 선택합니다.
- `followup`: `FOLLOWUP_EXPERIMENT`에서 `all`, `minimizer_seed_search`, `enhancer_target_only`를 선택합니다.
- `benchmark`: `BENCHMARK_TEMPLATE`, `BENCHMARK_RECYCLES`, `BENCHMARK_SEEDS`를 지정합니다. `target`은 GFP만, `both`는 두 단백질의 개별 구조를 제공하며 체인 간 정답 거리는 가립니다.
- `full`: 자체 형광 측정 CSV 경로를 `ASSAY_CSV`에 입력합니다. CSV에는 `id,ratio,group,assay,source`와 `sequence` 또는 `pdb,entity`가 필요합니다. `ratio`는 동일 실험 기준 대비 형광 배율이며 `group`은 유사 결합체 그룹입니다. 경로가 비어 있으면 공개 대조군을 사용합니다.


In [ ]:
RUN_MODE = "diagnostics" #@param ["diagnostics", "full", "benchmark", "followup"]
DIAGNOSTIC_EXPERIMENT = "all" #@param ["all", "enhancer_split", "minimizer_docking"]
FOLLOWUP_EXPERIMENT = "all" #@param ["all", "minimizer_seed_search", "enhancer_target_only"]
BENCHMARK_TEMPLATE = "both" #@param ["target", "both"]
BENCHMARK_RECYCLES = 10 #@param {type:"integer"}
BENCHMARK_SEEDS = "0,1,2" #@param {type:"string"}
ASSAY_CSV = "" #@param {type:"string"}
jobs = build_jobs(checkout, RUN_MODE, DIAGNOSTIC_EXPERIMENT, FOLLOWUP_EXPERIMENT,
                  BENCHMARK_TEMPLATE, BENCHMARK_RECYCLES, BENCHMARK_SEEDS, ASSAY_CSV)
completed_runs = []
for job in jobs:
    print(job["label"])


## 실험 실행

상세 출력은 실험별 로그에 저장하고 실패 시 마지막 40줄을 표시합니다. 같은 세션의 설치 환경과 모델 가중치를 재사용합니다. NanoBodyBuilder2·MEGADOCK·RF3는 별도 환경에서 실행됩니다.


In [ ]:
run_jobs(checkout, revision, jobs, completed_runs)


## 결과 비교


In [ ]:
display_results(checkout, RUN_MODE, completed_runs)


## 결과 다운로드

구조·지표·실제 패키지 목록·로그를 ZIP으로 내려받습니다. 중간 실험이 실패해도 완료된 실험이 있으면 이 셀에서 보존할 수 있습니다.


In [ ]:
from google.colab import files
result_zip = package_results(revision, RUN_MODE, completed_runs)
print(result_zip)
files.download(str(result_zip))
